# Candidate Generator v2.3 — kontrollü Colab/local çalıştırıcı

Bu notebook önce yalnız **26 leaf × 10 görüntü = 260 görüntülük throughput shard** çalıştırır. Bu bir altyapı smoke testidir; kanser sınıflandırma başarısı değildir. Tam 130.000 görüntü retrieval, duplicate kararı, family, split ve model eğitimi bu notebook tarafından başlatılmaz.

Çalıştırıcı frozen config, yedi yöntem bağımlılığı, veri envanteri ve ResNet18 checkpoint hash'lerini görüntülere dokunmadan önce doğrular.

In [ ]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
print('Ortam:', 'Google Colab' if IN_COLAB else 'Local/Jupyter')

## 1. Yolları ayarla

Colab'da veri setini Drive'a koyduğun gerçek konuma göre yalnız `DATASET_ROOT` satırını değiştir. Çıktı klasörü yeni ve boş olmalıdır; mevcut sonuçların üzerine yazılmaz.

In [ ]:
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
if IN_COLAB:
    PROJECT_ROOT = Path('/content/NNGA')
    DATASET_ROOT = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer/Multi Cancer')  # Gerekirse değiştir
    OUTPUT_ROOT = Path('/content/drive/MyDrive/NNGA_OUTPUTS/v2_3_throughput_shard')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
    DATASET_ROOT = PROJECT_ROOT / 'archive' / 'Multi Cancer' / 'Multi Cancer'
    OUTPUT_ROOT = PROJECT_ROOT / 'artifacts' / 'audit' / 'candidate_generator_v2_3_throughput_shard_notebook'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
assert (PROJECT_ROOT / 'config' / 'candidate_generator_v2_3.json').is_file(), 'Proje/config bulunamadı'
assert DATASET_ROOT.is_dir(), 'DATASET_ROOT yanlış; veri klasörünü düzelt'
assert not OUTPUT_ROOT.exists(), 'OUTPUT_ROOT zaten var; yeni bir klasör adı seç'

## 2. Ortam kontrolü

Colab ortamında import hatası çıkarsa `INSTALL_REQUIREMENTS=True` yapıp bu hücreyi bir kez çalıştır. Kurulumdan sonra Colab runtime yeniden başlatma isteyebilir. Çalışma raporu gerçek Python/PyTorch/CUDA sürümlerini kaydeder.

In [ ]:
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)

import torch, torchvision, numpy, PIL
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available(), '| cihaz:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Sınırlı throughput shard'ını çalıştır

Beklenen girdi 260 görüntüdür. Çıktıda `run_report.json`, `hash_manifest.json`, görüntü indeksi, pHash dizileri, dört descriptor matrisi ve aday çift tablosu oluşur.

In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'src' / 'run_candidate_generator_v2_3_throughput_shard.py'),
    '--data-dir', str(DATASET_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--images-per-leaf', '10',
    '--batch-size', '8',
    '--workers', '4',
    '--retrieval-block-size', '256',
    '--seed', '42',
]
print(' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 4. Sonucu oku ve Codex'e geri ver

Aşağıdaki özetin durumu `...throughput_shard_complete` olmalıdır. İnceleme için en az `run_report.json` ve `hash_manifest.json` dosyalarını geri ver. Bir hata olduysa hücre çıktısını ve oluşmuş klasörün içeriğini silmeden paylaş.

In [ ]:
import json
report = json.loads((OUTPUT_ROOT / 'run_report.json').read_text(encoding='utf-8'))
print('Durum:', report['status'])
print('Görüntü:', report['input']['shard_image_count'])
print('Cihaz:', report['environment']['cuda_device'])
print('Descriptor süresi:', round(report['timing_seconds']['robust_descriptor_generation'], 2), 's')
print('Aday birleşimi:', report['candidate_counts']['union_pairs'])
print('Çift/görüntü:', round(report['candidate_counts']['union_pairs_per_image'], 3))
print('\nCodex’e verilecek dosyalar:')
print(OUTPUT_ROOT / 'run_report.json')
print(OUTPUT_ROOT / 'hash_manifest.json')